In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from estimation_forecast_functions_var import DataCleaner_HKD
import holidays
from simple_strat_funcs import Clean_Implied_Vols_HKD_with_smile
from scipy.stats import norm
from hedging_strategy_class_NEW_KURT_SKEW_vega_VAR import Compare_Trading_Strategies
import seaborn as sns

/Users/alexvillamartin/Documents/MSc Diss/Code/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Global params

In [2]:
train_size = 0.7
test_size = 1 - train_size

TICKER = "USDUKD"

# always use same amount of money in USD to start and convert where necessary
IC_USD = 1_000_000 # initial capital in USD
current_rate = 7.85
IC_HKD = IC_USD * current_rate
NOTIONAL_USD = 1_000_000 # this is used in strategy as our notional to ensure trade positions are in domestic currency

MAX_DELTA_DIFF = 500 # this param doesnt change much as our delta positions are usually much larger
SIGNAL_LB = 0.01
SIGNAL_UB = 0.99
TRANSACTION_COST_BOOL = True
TRANSACTION_COSTS_SPOT = 0.0002 / 2 # for half a leg ie selling/buying once not to enter and exit the trade
TRANSACTION_COSTS_OPTION = 0.0005 / 2 # [change to vol spreads later] for half a leg again, both in decimals

HYPERPARAM_SORT = 'sharpe_ratio' # can be 'sharpe_ratio', 'cagr', 'max_drawdown', 'total_return'
GARCH_1_2_INDICATOR = False     # FALSE due to better results despite BIC selection
FIGARCH_0_INDICATOR = True

k_bar_MSM = 8
b_MSM = 2.0
gamma_kbar_MSM = 0.5

M = 300 # figarch lags

CONVERT_USD_INDICATOR = True # convert all portfolio values and metrics to USD for fair comparison - use when USD is not domestic

long_threshs = np.array([1.02, 1.03, 1.04, 1.05, 1.06, 1.07,  1.08, 1.09, 1.10, 1.11, 1.12, 1.13, 1.14, 1.15, 1.16, 1.17, 1.18, 1.19, 1.20, 1.21, 1.22, 1.23, 1.24, 1.25, 1.26, 1.27])
short_threshs = np.array([0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92,  0.91, 0.90, 0.89, 0.88, 0.87, 0.86, 0.85, 0.84, 0.83, 0.82, 0.81, 0.80, 0.79, 0.78, 0.77, 0.76, 0.75, 0.74, 0.73])
sig_multipliers = np.array([1, 3, 5, 7, 9, 11, 13, 15, 17, 20])


# Global data

In [3]:
usdhkd_5m = pd.read_parquet("USD_HKD_5M.parquet").copy()[['c']]
usdhkd_5m.rename(columns={'c': 'spot'}, inplace=True)  
cleaner_usdhkd = DataCleaner_HKD(usdhkd_5m, start_hr=2, end_hr=18, unit_test=False) 
realised_variance_usdhkd = cleaner_usdhkd.clean_data()
daily_log_returns_usdhkd = pd.read_parquet("DF_D_USDHKD.parquet")['Log_r']
start_date = pd.to_datetime(realised_variance_usdhkd.index.min())
end_date = pd.to_datetime(realised_variance_usdhkd.index.max())
spot_curr = pd.read_parquet("/Users/alexvillamartin/Documents/MSc Diss/Code/DF_D_USDHKD.parquet")
overnight_domestic_rate = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/honia_overnight.csv").set_index("date") 
overnight_domestic_rate.index = (pd.to_datetime(overnight_domestic_rate.index, format='mixed').normalize())
overnight_foreign_rate = pd.read_csv("SOFR_daily.csv").set_index("date")

# Global functions

Change holidays where needed. 

In [4]:
def align_spots(df, aligned_df, start, end):

    df = df[(df.index >= start) & (df.index <= end)]

    years = range(start.year, end.year + 1)

    first = holidays.US(years=years)
    second = holidays.HK(years=years)

    hols = pd.to_datetime(list(set(first) | set(second))).normalize()

    idx  = df.index
    mask = ~idx.isin(hols)
    new_df = df.loc[mask]

    df_aligned_dates = aligned_df.index
    common_dates = new_df.index.intersection(df_aligned_dates)
    new_df = new_df.loc[common_dates]

    return new_df

def sort_rates(r_b, r_t, overn_r, aligned_df, overn_for_r):

    r_b.index = pd.to_datetime(r_b.index)
    valid_mask = ~( 
                   r_t.index.isna())
    r_t_clean = r_t[valid_mask]
    r_t_clean.index = pd.to_datetime(r_t_clean.index, format='%Y-%m-%d')
    
    valid_mask2 = ~(
                   overn_r.index.isna())
    overn_r_clean = overn_r[valid_mask2]
    overn_r_clean.index = pd.to_datetime(overn_r_clean.index, format='%Y-%m-%d')

    valid_mask3 = ~(
                   overn_for_r.index.isna())
    overn_for_r_clean = overn_for_r[valid_mask3]
    overn_for_r_clean.index = pd.to_datetime(overn_for_r_clean.index, format='mixed').normalize()    

    r_b = r_b.reindex(aligned_df.index)
    r_t_clean = r_t_clean.reindex(aligned_df.index)
    overn_r_clean = overn_r_clean.reindex(aligned_df.index)
    overn_for_r_clean = overn_for_r_clean.reindex(aligned_df.index)

    r_b = r_b.ffill()
    r_t_clean = r_t_clean.ffill()
    overn_r_clean = overn_r_clean.ffill()
    overn_for_r_clean = overn_for_r_clean.ffill()

    # convert into decimals and continously compunded versions for BSE
    r_b = pd.to_numeric(r_b['rate_pct'], errors='coerce')
    r_t_clean = pd.to_numeric(r_t_clean['rate_pct'], errors='coerce')
    overn_r_clean = pd.to_numeric(overn_r_clean['rate'], errors='coerce')
    r_b = r_b / 100
    r_t_clean = r_t_clean / 100
    overn_r_clean = overn_r_clean / 100 # not continously compounded
    overn_r_clean *= 1/365 # daily

    overn_for_r_clean = pd.to_numeric(overn_for_r_clean['value'], errors='coerce')
    overn_for_r_clean = overn_for_r_clean / 100 # not continously compounded
    overn_for_r_clean *= 1/365 # daily

    r_b = np.log(1 + r_b)
    r_t_clean = np.log(1 + r_t_clean)

    return r_b, r_t_clean, overn_r_clean, overn_for_r_clean

# H=30, T=1Mo

In [5]:
OPTION_MATURITY_1 = 1 / 12
H_1 = 30 

implied_vol_data_1 = pd.read_excel('/Users/alexvillamartin/Documents/MSc Diss/Code/USDHKD_1MO_ATM_D.xlsx')
vol_smile_data_1 = pd.read_csv("usdhkd_vol_smile_1mo_extra.csv").set_index("CalculationDate")
Data_clean_1 = Clean_Implied_Vols_HKD_with_smile(data=implied_vol_data_1, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdhkd, 
                                align_df2=daily_log_returns_usdhkd, 
                                smile_df=vol_smile_data_1)
implied_vol_data_1, realised_variance_1, daily_log_returns_1, vol_smile_data_1 = Data_clean_1.get_clean_data()

N_1 = len(daily_log_returns_1)
test_align_1 = daily_log_returns_1.iloc[N_1//2:-H_1]
spot_curr_test_1 = align_spots(spot_curr, test_align_1, start_date, end_date)

r_b_1 = pd.read_csv("SOFR_1mo_compounded.csv").set_index("date")
r_t_1 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/honia_1mo_compounded.csv").set_index("date")[['rate_pct']]

r_b_test_1, r_t_test_1, overn_dom_r_test_1, overn_for_r_test_1= sort_rates(r_b_1, r_t_1, overnight_domestic_rate, test_align_1, overnight_foreign_rate) 
r_b_test_1.ffill(inplace=True)

In [6]:
strategy_1 = Compare_Trading_Strategies(
    return_series=daily_log_returns_1, 
    realised_variance_series=realised_variance_1,
    atm_implied_vol_data=implied_vol_data_1,
    vol_smile_data=vol_smile_data_1,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_HKD,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_1,
    forecast_horizon=H_1,
    spot_series=spot_curr_test_1,
    overnight_domestic_rate=overn_dom_r_test_1,
    overnight_foreign_rate=overn_for_r_test_1,
    domestic_rate=r_t_test_1,
    foreign_rate=r_b_test_1, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers,
    FIGARCH_ind=FIGARCH_0_INDICATOR)

strategy_1.prepare_universal_series()
strategy_1.get_BMSM_data()
strategy_1.get_GARCH_data()
strategy_1.get_FIGARCH_data()

Estimated parameters: m0=1.577052e+00, sigma_bar=2.756652e-02
Final log-likelihood: 2.767892e+03
Estimated parameters: m0=1.579971e+00, sigma_bar=4.025620e-02
Final log-likelihood: 4.653722e+03
Estimated parameters: omega=4.771622502566933e-05, alpha=0.1510, beta=0.8257
Estimated parameters: omega=3.098029696817893e-05, alpha=0.1525, beta=0.8335
Estimated parameters: omega=0.0001827494442326697, d=0.5511
Final log-likelihood = 7170.5390
Estimated parameters: omega=0.00014476551971041553, d=0.5175
Final log-likelihood = 12155.5965


In [7]:
error_metrics_df_1, log_ls, m_z_results, se_results = strategy_1.in_sample_predictions()

In [12]:
error_metrics_df_1

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.641628,NaN,NaN,0.664871,NaN,NaN
BMSM OLS,0.732157,0.594173,0.723745,0.965979,3.777464,0.999917
GARCH,0.748038,NaN,NaN,0.822396,NaN,NaN
GARCH OLS,0.876298,1.296176,0.902415,1.066137,3.272497,0.999451
FIGARCH,0.818763,NaN,NaN,0.909685,NaN,NaN
FIGARCH OLS,1.251215,2.065618,0.980458,1.180015,5.046262,1.000000


In [13]:
log_ls

{'BMSM': np.float64(2767.891547984952),
 'GARCH': np.float64(7042.563304366555),
 'FIGARCH': np.float64(7170.539013349684)}

In [14]:
m_z_results 

{'BMSM': {'alpha_hat': 2.1618403970574283e-05,
  'beta_hat': 0.8682291919082918,
  'alpha_p': 0.012241966015318607,
  'beta_p': 0.48992488873123186},
 'GARCH': {'alpha_hat': 2.484673967696388e-05,
  'beta_hat': 0.6660792558185356,
  'alpha_p': 2.5848168407326875e-05,
  'beta_p': 0.002602979759719408},
 'FIGARCH': {'alpha_hat': 2.0941831200679637e-05,
  'beta_hat': 0.569600245578273,
  'alpha_p': 0.0005090141781987139,
  'beta_p': 1.7757381745126098e-07}}

In [15]:
se_results

{'BMSM': array([0.02315777, 0.00362436]),
 'GARCH': array([0.00100001, 0.01091731, 0.01153539]),
 'FIGARCH': array([1.86554419e-05, 2.53469345e-02])}

In [16]:
strategy_1.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.108
Model:                            OLS   Adj. R-squared:                  0.105
Method:                 Least Squares   F-statistic:                     3.344
Date:                Sat, 30 Aug 2025   Prob (F-statistic):            0.00984
Time:                        22:02:32   Log-Likelihood:                -1672.2
No. Observations:                1173   AIC:                             3354.
Df Residuals:                    1168   BIC:                             3380.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.2387      0.075      3.196      0.001       0.092       0.385
x1            -0.1089      0.120     -0.905      0.365      -0.344       0.127
x2            -0.2698      0.106     -2.538      0.011      -0.478      -0.061
x3             0.1096      0.109      1.009      0.313      -0.103       0.323
x4            -0.0125      0.116     -0.108      0.914      -0.241       0.215
==============================================================================
Omnibus:                       49.220   Durbin-Watson:                   0.046
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               85.564
Skew:                           0.323   Prob(JB):                     2.63e-19
Kurtosis:                       4.155   Cond. No.                         3.23
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [17]:
strategy_1.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.013
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     3.859
Date:                Sat, 30 Aug 2025   Prob (F-statistic):            0.00920
Time:                        22:02:36   Log-Likelihood:                -1649.7
No. Observations:                1173   AIC:                             3307.
Df Residuals:                    1169   BIC:                             3328.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0640      0.073     -0.872      0.383      -0.208       0.080
x1            -0.0584      0.030     -1.976      0.048      -0.116      -0.000
x2             0.0188      0.079      0.238      0.812      -0.136       0.174
x3             0.0743      0.088      0.844      0.399      -0.098       0.247
==============================================================================
Omnibus:                       25.675   Durbin-Watson:                   0.052
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               46.579
Skew:                           0.129   Prob(JB):                     7.68e-11
Kurtosis:                       3.942   Cond. No.                         1.80
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [18]:
strategy_1.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                    0.8326
Date:                Sat, 30 Aug 2025   Prob (F-statistic):              0.476
Time:                        22:02:40   Log-Likelihood:                -1643.3
No. Observations:                1173   AIC:                             3295.
Df Residuals:                    1169   BIC:                             3315.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.3413      0.073     -4.699      0.000      -0.484      -0.199
x1             0.0347      0.075      0.462      0.644      -0.112       0.182
x2             0.0083      0.096      0.086      0.931      -0.180       0.197
x3             0.1106      0.109      1.016      0.310      -0.103       0.324
==============================================================================
Omnibus:                       37.634   Durbin-Watson:                   0.057
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               65.789
Skew:                           0.245   Prob(JB):                     5.18e-15
Kurtosis:                       4.052   Cond. No.                         2.40
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=91, T=3MO

In [19]:
OPTION_MATURITY_2 = 3 / 12
H_2 = 91 

vol_smile_data_2 = pd.read_csv("usdhkd_vol_smile_3mo_extra.csv").set_index("CalculationDate")
implied_vol_data_2 = pd.DataFrame({
    'Exchange Date': vol_smile_data_2.index, 
    "Bid": vol_smile_data_2['ATM'], 
    "Ask": vol_smile_data_2['ATM'],
    "BidNet": vol_smile_data_2['ATM']})
Data_clean_2 = Clean_Implied_Vols_HKD_with_smile(data=implied_vol_data_2, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdhkd, 
                                align_df2=daily_log_returns_usdhkd, 
                                smile_df=vol_smile_data_2)
implied_vol_data_2, realised_variance_2, daily_log_returns_2, vol_smile_data_2 = Data_clean_2.get_clean_data()

N_2 = len(daily_log_returns_2)
test_align_2 = daily_log_returns_2.iloc[N_2//2:-H_2]
spot_curr_test_2 = align_spots(spot_curr, test_align_2, start_date, end_date)

r_b_2 = pd.read_csv("SOFR_3mo_compounded.csv").set_index("date")
r_t_2 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/honia_3mo_compounded.csv").set_index("date")[['rate_pct']]

r_b_test_2, r_t_test_2, overn_dom_r_test_2, overn_for_r_test_2= sort_rates(r_b_2, r_t_2, overnight_domestic_rate, test_align_2, overnight_foreign_rate) 
r_b_test_2.ffill(inplace=True)

In [20]:
strategy_2 = Compare_Trading_Strategies(
    return_series=daily_log_returns_2, 
    realised_variance_series=realised_variance_2,
    atm_implied_vol_data=implied_vol_data_2,
    vol_smile_data=vol_smile_data_2,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_HKD,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_2,
    forecast_horizon=H_2,
    spot_series=spot_curr_test_2,
    overnight_domestic_rate=overn_dom_r_test_2,
    overnight_foreign_rate=overn_for_r_test_2,
    domestic_rate=r_t_test_2,
    foreign_rate=r_b_test_2, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers,
    FIGARCH_ind=FIGARCH_0_INDICATOR)

strategy_2.prepare_universal_series()
strategy_2.get_BMSM_data()
strategy_2.get_GARCH_data()
strategy_2.get_FIGARCH_data()

Estimated parameters: m0=1.577052e+00, sigma_bar=2.756652e-02
Final log-likelihood: 2.767892e+03
Estimated parameters: m0=1.585861e+00, sigma_bar=4.076489e-02
Final log-likelihood: 4.597289e+03
Estimated parameters: omega=4.771622502566933e-05, alpha=0.1510, beta=0.8257
Estimated parameters: omega=7.168311789386581e-05, alpha=0.3299, beta=0.6597
Estimated parameters: omega=0.0001827494442326697, d=0.5511
Final log-likelihood = 7170.5390
Estimated parameters: omega=0.00014511476849743838, d=0.5233
Final log-likelihood = 11962.5046


In [21]:
error_metrics_df_2, _, m_z_results_2, _ = strategy_2.in_sample_predictions()

In [22]:
error_metrics_df_2

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.698819,NaN,NaN,0.709700,NaN,NaN
BMSM OLS,0.432844,-0.914681,0.180279,0.668218,-0.278428,0.390368
GARCH,0.934122,NaN,NaN,0.877902,NaN,NaN
GARCH OLS,0.886909,-0.230687,0.408800,0.956486,0.751944,0.773878
FIGARCH,1.396558,NaN,NaN,1.167663,NaN,NaN
FIGARCH OLS,0.784430,-1.607408,0.054125,0.876465,-2.247231,0.012410


In [23]:
m_z_results_2

{'BMSM': {'alpha_hat': 3.899637621524714e-05,
  'beta_hat': 0.5639536656628357,
  'alpha_p': 2.7903525439441255e-05,
  'beta_p': 0.05178042619351868},
 'GARCH': {'alpha_hat': 3.913999454177084e-05,
  'beta_hat': 0.36523901905572637,
  'alpha_p': 1.1956415890903109e-09,
  'beta_p': 2.0455251566044503e-10},
 'FIGARCH': {'alpha_hat': 3.768633993450511e-05,
  'beta_hat': 0.2539814470980563,
  'alpha_p': 4.8204129209019333e-08,
  'beta_p': 1.1536562531438846e-30}}

In [24]:
strategy_2.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.354
Model:                            OLS   Adj. R-squared:                  0.352
Method:                 Least Squares   F-statistic:                     21.62
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           3.41e-17
Time:                        22:05:21   Log-Likelihood:                -1350.8
No. Observations:                1112   AIC:                             2712.
Df Residuals:                    1107   BIC:                             2737.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.5369      0.064      8.414      0.000       0.412       0.662
x1            -0.5083      0.087     -5.875      0.000      -0.678      -0.339
x2            -0.1895      0.081     -2.350      0.019      -0.348      -0.031
x3             0.1315      0.047      2.826      0.005       0.040       0.223
x4            -0.1632      0.064     -2.539      0.011      -0.289      -0.037
==============================================================================
Omnibus:                      303.950   Durbin-Watson:                   0.022
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               73.481
Skew:                          -0.362   Prob(JB):                     1.11e-16
Kurtosis:                       1.970   Cond. No.                         3.12
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [25]:
strategy_2.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.017
Model:                            OLS   Adj. R-squared:                  0.014
Method:                 Least Squares   F-statistic:                     6.176
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           0.000367
Time:                        22:05:23   Log-Likelihood:                -1370.8
No. Observations:                1112   AIC:                             2750.
Df Residuals:                    1108   BIC:                             2770.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0055      0.065     -0.085      0.932      -0.132       0.121
x1            -0.1090      0.028     -3.923      0.000      -0.164      -0.055
x2             0.0245      0.052      0.475      0.635      -0.077       0.126
x3             0.0027      0.067      0.040      0.968      -0.129       0.134
==============================================================================
Omnibus:                      187.595   Durbin-Watson:                   0.032
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               44.424
Skew:                           0.120   Prob(JB):                     2.26e-10
Kurtosis:                       2.051   Cond. No.                         1.30
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [26]:
strategy_2.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.048
Model:                            OLS   Adj. R-squared:                  0.045
Method:                 Least Squares   F-statistic:                     6.733
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           0.000168
Time:                        22:05:24   Log-Likelihood:                -1373.6
No. Observations:                1112   AIC:                             2755.
Df Residuals:                    1108   BIC:                             2775.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.3866      0.065     -5.905      0.000      -0.515      -0.258
x1            -0.2010      0.049     -4.130      0.000      -0.296      -0.106
x2             0.0478      0.051      0.932      0.351      -0.053       0.148
x3            -0.0162      0.067     -0.240      0.810      -0.148       0.116
==============================================================================
Omnibus:                      204.234   Durbin-Watson:                   0.009
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               44.362
Skew:                           0.071   Prob(JB):                     2.33e-10
Kurtosis:                       2.032   Cond. No.                         1.76
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=182, T=6MO

In [27]:
OPTION_MATURITY_3 = 6 / 12
H_3 = 182

vol_smile_data_3 = pd.read_csv("usdhkd_vol_smile_6mo_extra.csv").set_index("CalculationDate")
implied_vol_data_3 = pd.DataFrame({
    'Exchange Date': vol_smile_data_3.index, 
    "Bid": vol_smile_data_3['ATM'], 
    "Ask": vol_smile_data_3['ATM'],
    "BidNet": vol_smile_data_3['ATM']})
Data_clean_3 = Clean_Implied_Vols_HKD_with_smile(data=implied_vol_data_3, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdhkd, 
                                align_df2=daily_log_returns_usdhkd, 
                                smile_df=vol_smile_data_3)
implied_vol_data_3, realised_variance_3, daily_log_returns_3, vol_smile_data_3 = Data_clean_3.get_clean_data()

N_3 = len(daily_log_returns_3)
test_align_3 = daily_log_returns_3.iloc[N_3//2:-H_3]
spot_curr_test_3 = align_spots(spot_curr, test_align_3, start_date, end_date)

r_b_3 = pd.read_csv("SOFR_6mo_compounded.csv").set_index("date")
r_t_3 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/honia_6mo_compounded.csv").set_index("date")[['rate_pct']]

r_b_test_3, r_t_test_3, overn_dom_r_test_3, overn_for_r_test_3= sort_rates(r_b_3, r_t_3, overnight_domestic_rate, test_align_3, overnight_foreign_rate) 
r_b_test_3.ffill(inplace=True)

In [31]:
strategy_3 = Compare_Trading_Strategies(
    return_series=daily_log_returns_3, 
    realised_variance_series=realised_variance_3,
    atm_implied_vol_data=implied_vol_data_3,
    vol_smile_data=vol_smile_data_3,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_HKD,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_3,
    forecast_horizon=H_3,
    spot_series=spot_curr_test_3,
    overnight_domestic_rate=overn_dom_r_test_3,
    overnight_foreign_rate=overn_for_r_test_3,
    domestic_rate=r_t_test_3,
    foreign_rate=r_b_test_3, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers, 
    FIGARCH_ind=FIGARCH_0_INDICATOR)

strategy_3.prepare_universal_series()
strategy_3.get_BMSM_data()
strategy_3.get_GARCH_data()
strategy_3.get_FIGARCH_data()

Estimated parameters: m0=1.577052e+00, sigma_bar=2.756652e-02
Final log-likelihood: 2.767892e+03
Estimated parameters: m0=1.593911e+00, sigma_bar=4.063711e-02
Final log-likelihood: 4.521357e+03
Estimated parameters: omega=4.771622502566933e-05, alpha=0.1510, beta=0.8257
Estimated parameters: omega=7.685684282269831e-05, alpha=0.3487, beta=0.6404
Estimated parameters: omega=0.0001827494442326697, d=0.5511
Final log-likelihood = 7170.5390
Estimated parameters: omega=0.00014733496101059914, d=0.5381
Final log-likelihood = 11690.7960


In [32]:
error_metrics_df_3, _, m_z_results_3, _ = strategy_3.in_sample_predictions()

In [33]:
error_metrics_df_3 

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.728507,NaN,NaN,0.797639,NaN,NaN
BMSM OLS,0.429338,-0.806422,0.210094,0.663802,-0.534239,0.296646
GARCH,0.694479,NaN,NaN,0.772178,NaN,NaN
GARCH OLS,0.901910,0.758796,0.775925,0.968811,0.843740,0.800494
FIGARCH,1.877748,NaN,NaN,1.405951,NaN,NaN
FIGARCH OLS,0.872245,-2.096725,0.018132,0.959389,-1.741314,0.040965


In [34]:
m_z_results_3

{'BMSM': {'alpha_hat': 3.846591334730861e-05,
  'beta_hat': 0.7655502641131466,
  'alpha_p': 3.131075953384961e-06,
  'beta_p': 0.3361702753508139},
 'GARCH': {'alpha_hat': 3.4466278922539835e-05,
  'beta_hat': 0.49568103426264015,
  'alpha_p': 4.641218586180261e-07,
  'beta_p': 5.264513965708125e-06},
 'FIGARCH': {'alpha_hat': 3.615665417801017e-05,
  'beta_hat': 0.2558060705772852,
  'alpha_p': 6.79046479172038e-09,
  'beta_p': 1.131421071215194e-48}}

In [35]:
strategy_3.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.453
Model:                            OLS   Adj. R-squared:                  0.451
Method:                 Least Squares   F-statistic:                     48.48
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           2.40e-37
Time:                        22:11:31   Log-Likelihood:                -821.01
No. Observations:                1021   AIC:                             1652.
Df Residuals:                    1016   BIC:                             1677.
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.7639      0.044     17.200      0.000       0.677       0.851
x1            -0.3648      0.060     -6.089      0.000      -0.482      -0.247
x2            -0.1306      0.052     -2.517      0.012      -0.232      -0.029
x3            -0.0370      0.052     -0.716      0.474      -0.138       0.064
x4             0.0385      0.043      0.894      0.371      -0.046       0.123
==============================================================================
Omnibus:                       91.740   Durbin-Watson:                   0.013
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              105.826
Skew:                          -0.759   Prob(JB):                     1.05e-23
Kurtosis:                       2.575   Cond. No.                         3.11
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [36]:
strategy_3.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.089
Model:                            OLS   Adj. R-squared:                  0.086
Method:                 Least Squares   F-statistic:                     22.66
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           3.31e-14
Time:                        22:11:33   Log-Likelihood:                -840.39
No. Observations:                1021   AIC:                             1689.
Df Residuals:                    1017   BIC:                             1709.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0631      0.045      1.400      0.162      -0.025       0.151
x1            -0.0778      0.020     -3.955      0.000      -0.116      -0.039
x2            -0.0753      0.054     -1.404      0.160      -0.180       0.030
x3             0.0972      0.042      2.327      0.020       0.015       0.179
==============================================================================
Omnibus:                       84.067   Durbin-Watson:                   0.029
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               63.166
Skew:                          -0.510   Prob(JB):                     1.92e-14
Kurtosis:                       2.334   Cond. No.                         1.37
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [37]:
strategy_3.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.208
Model:                            OLS   Adj. R-squared:                  0.206
Method:                 Least Squares   F-statistic:                     20.19
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.02e-12
Time:                        22:11:35   Log-Likelihood:                -856.90
No. Observations:                1021   AIC:                             1722.
Df Residuals:                    1017   BIC:                             1742.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.4500      0.046     -9.757      0.000      -0.540      -0.360
x1            -0.2334      0.040     -5.877      0.000      -0.311      -0.156
x2            -0.0502      0.052     -0.972      0.331      -0.151       0.051
x3             0.0759      0.044      1.724      0.085      -0.010       0.162
==============================================================================
Omnibus:                      111.130   Durbin-Watson:                   0.007
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               95.640
Skew:                          -0.669   Prob(JB):                     1.71e-21
Kurtosis:                       2.322   Cond. No.                         1.53
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

# H=364, T=1Y

In [38]:
OPTION_MATURITY_4 = 1
H_4 = 364

vol_smile_data_4 = pd.read_csv("usdhkd_vol_smile_1y_extra.csv").set_index("CalculationDate")
implied_vol_data_4 = pd.DataFrame({
    'Exchange Date': vol_smile_data_4.index, 
    "Bid": vol_smile_data_4['ATM'], 
    "Ask": vol_smile_data_4['ATM'],
    "BidNet": vol_smile_data_4['ATM']})
Data_clean_4 = Clean_Implied_Vols_HKD_with_smile(data=implied_vol_data_4, 
                                start_date=start_date, 
                                end_date=end_date, 
                                align_df1=realised_variance_usdhkd, 
                                align_df2=daily_log_returns_usdhkd, 
                                smile_df=vol_smile_data_4)
implied_vol_data_4, realised_variance_4, daily_log_returns_4, vol_smile_data_4 = Data_clean_4.get_clean_data()

N_4 = len(daily_log_returns_4)
test_align_4 = daily_log_returns_4.iloc[N_4//2:-H_4]
spot_curr_test_4 = align_spots(spot_curr, test_align_4, start_date, end_date)

r_b_4 = pd.read_csv("SOFR_1y_compounded.csv").set_index("date")
r_t_4 = pd.read_csv("/Users/alexvillamartin/Documents/MSc Diss/Code/honia_1y_compounded.csv").set_index("date")[['rate_pct']]

r_b_test_4, r_t_test_4, overn_dom_r_test_4, overn_for_r_test_4= sort_rates(r_b_4, r_t_4, overnight_domestic_rate, test_align_4, overnight_foreign_rate) 
r_b_test_4.ffill(inplace=True)

In [39]:
strategy_4 = Compare_Trading_Strategies(
    return_series=daily_log_returns_4, 
    realised_variance_series=realised_variance_4,
    atm_implied_vol_data=implied_vol_data_4,
    vol_smile_data=vol_smile_data_4,
    train_size=train_size,
    ticker=TICKER,
    initial_capital_domestic=IC_HKD,
    notional_base=NOTIONAL_USD,
    maximum_delta_difference=MAX_DELTA_DIFF,
    signal_lb=SIGNAL_LB,
    signal_ub=SIGNAL_UB,
    transaction_cost_indicator=TRANSACTION_COST_BOOL,
    transaction_costs_spot=TRANSACTION_COSTS_SPOT,
    transaction_costs_option=TRANSACTION_COSTS_OPTION,
    option_maturity=OPTION_MATURITY_4,
    forecast_horizon=H_4,
    spot_series=spot_curr_test_4,
    overnight_domestic_rate=overn_dom_r_test_4,
    overnight_foreign_rate=overn_for_r_test_4,
    domestic_rate=r_t_test_4,
    foreign_rate=r_b_test_4, 
    plots = False, 
    verbose=False, 
    sort_hyperparams_by=HYPERPARAM_SORT, 
    garch_1_2_indicator=GARCH_1_2_INDICATOR, 
    kbar=k_bar_MSM,
    b=b_MSM,
    gamma_kbar=gamma_kbar_MSM,
    convert_USD=CONVERT_USD_INDICATOR, 
    M = M, 
    long_thresholds=long_threshs,
    short_thresholds=short_threshs,
    sig_multipliers=sig_multipliers, 
    FIGARCH_ind=FIGARCH_0_INDICATOR)

strategy_4.prepare_universal_series()
strategy_4.get_BMSM_data()
strategy_4.get_GARCH_data()
strategy_4.get_FIGARCH_data()


Estimated parameters: m0=1.577052e+00, sigma_bar=2.756652e-02
Final log-likelihood: 2.767892e+03
Estimated parameters: m0=1.555780e+00, sigma_bar=2.667106e-02
Final log-likelihood: 4.303206e+03
Estimated parameters: omega=4.771622502566933e-05, alpha=0.1510, beta=0.8257
Estimated parameters: omega=2.110054089742415e-05, alpha=0.1532, beta=0.8372
Estimated parameters: omega=0.0001827494442326697, d=0.5511
Final log-likelihood = 7170.5390
Estimated parameters: omega=9.98760263842816e-05, d=0.5128
Final log-likelihood = 11129.3053


In [40]:
error_metrics_df_4, _, m_z_results_4, _ = strategy_4.in_sample_predictions()

In [41]:
error_metrics_df_4

,Norm MSE,DM Test Stat MSE,DM p-value (one-sided) MSE,Norm MAE,DM Test Stat MAE,DM p-value (one-sided) MAE
BMSM,0.832982,NaN,NaN,0.909366,NaN,NaN
BMSM OLS,0.303112,-1.398510,0.081165,0.518639,-1.403871,0.080364
GARCH,0.533689,NaN,NaN,0.752276,NaN,NaN
GARCH OLS,0.668635,1.260971,0.896165,0.778943,0.209345,0.582885
FIGARCH,3.313964,NaN,NaN,1.838775,NaN,NaN
FIGARCH OLS,0.680206,-1.867922,0.031061,0.793088,-1.663081,0.048335


In [42]:
m_z_results_4

{'BMSM': {'alpha_hat': 3.096078963360143e-05,
  'beta_hat': 1.361121084248443,
  'alpha_p': 7.417123489857954e-05,
  'beta_p': 0.1979181446779028},
 'GARCH': {'alpha_hat': 3.529732341911542e-05,
  'beta_hat': 0.5334628176055569,
  'alpha_p': 4.910464312018649e-05,
  'beta_p': 0.002112560057561085},
 'FIGARCH': {'alpha_hat': 4.281973069163782e-05,
  'beta_hat': 0.17686773712692183,
  'alpha_p': 5.17901338408347e-12,
  'beta_p': 1.3118252807097087e-78}}

In [43]:
strategy_4.model_bmsm.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.806
Model:                            OLS   Adj. R-squared:                  0.805
Method:                 Least Squares   F-statistic:                     117.6
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           1.58e-79
Time:                        22:16:14   Log-Likelihood:                 309.46
No. Observations:                 839   AIC:                            -608.9
Df Residuals:                     834   BIC:                            -585.3
Df Model:                           4                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.8596      0.015     57.624      0.000       0.830       0.889
x1            -0.2108      0.029     -7.369      0.000      -0.267      -0.155
x2            -0.0438      0.025     -1.782      0.075      -0.092       0.004
x3            -0.0658      0.016     -4.099      0.000      -0.097      -0.034
x4             0.1743      0.017     10.333      0.000       0.141       0.207
==============================================================================
Omnibus:                       35.496   Durbin-Watson:                   0.037
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               34.295
Skew:                           0.449   Prob(JB):                     3.57e-08
Kurtosis:                       2.583   Cond. No.                         3.23
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [44]:
strategy_4.model_garch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.681
Model:                            OLS   Adj. R-squared:                  0.680
Method:                 Least Squares   F-statistic:                     101.0
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           8.93e-56
Time:                        22:16:16   Log-Likelihood:                 304.86
No. Observations:                 839   AIC:                            -601.7
Df Residuals:                     835   BIC:                            -582.8
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0299      0.015      2.029      0.042       0.001       0.059
x1            -0.0410      0.009     -4.528      0.000      -0.059      -0.023
x2            -0.0932      0.013     -7.142      0.000      -0.119      -0.068
x3             0.1718      0.016     10.502      0.000       0.140       0.204
==============================================================================
Omnibus:                        7.244   Durbin-Watson:                   0.116
Prob(Omnibus):                  0.027   Jarque-Bera (JB):                6.446
Skew:                           0.155   Prob(JB):                       0.0398
Kurtosis:                       2.702   Cond. No.                         1.88
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""

In [45]:
strategy_4.model_figarch.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.734
Model:                            OLS   Adj. R-squared:                  0.733
Method:                 Least Squares   F-statistic:                     76.99
Date:                Sat, 30 Aug 2025   Prob (F-statistic):           5.67e-44
Time:                        22:16:17   Log-Likelihood:                 269.48
No. Observations:                 839   AIC:                            -531.0
Df Residuals:                     835   BIC:                            -512.0
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.6811      0.016    -43.225      0.000      -0.712      -0.650
x1            -0.1274      0.021     -6.038      0.000      -0.169      -0.086
x2            -0.0858      0.016     -5.373      0.000      -0.117      -0.055
x3             0.1707      0.018      9.747      0.000       0.136       0.205
==============================================================================
Omnibus:                       28.795   Durbin-Watson:                   0.030
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               19.281
Skew:                           0.247   Prob(JB):                     6.50e-05
Kurtosis:                       2.446   Cond. No.                         2.02
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 6 lags and without small sample correction
"""